# The <span class="uvk">uvk</span> IPython extension

The goal of the IPython kernel extension is to provide a tool for creating and editing a notebook's
metadata cell.
This tool is a single magic function named [`%uvk`](#uvk-reference).

## The notebook's metadata cell

While a notebook can carry multiple metadata snippets across its cells,
only one is retained by <span class="uvk">uvk</span> when it prepares the environment in which to launch the kernel.

<div class="admonition important">
    <div class="admonition-title">The metadata cell</div>
    <p>
        The metadata snippet will be parsed out of the first code cell that contains a script metadata header:
    </p>
    <div class="highlight">
        <pre>#&nbsp;///&nbsp;script</pre>
    </div>
    <p>
        Once it has found such a cell,
        <span class="uvk">uvk</span> attempts to parse the complete metadata snippet from the cell.
        If this parsing fails, the whole kernel fails to start.
    </p>
</div>

No cell scanned beyond the first cell where a script metadata header is encountered.
Valid metadata cells beyond this first broken one are useless.
Similarly, valid metadata cells beyond one whose parsing succeeds *are ignored*.
Thus,
whenever the package requirements are tweaked,
and one observes no change in the package loadout when restarting the kernel,
one should
<strong>go check that there's no script metadata cell above</strong>
the one being modified.

## Loading the extension

In [1]:
%load_ext uvk

In [3]:
%uvk --help-rtd

## <a id="uvk-reference"></a>The `%uvk`/`%%uvk` magic

Assist in the editing of various aspects of a cell containing inline script
metadata. It is not strictly necessary to use this cell magic to change the
metadata, but since uvk's logic for detecting and parsing script metadata is
strict, and since comment lines and TOML syntax can be tedious to wrangle,
this tool might help.

This magic can be used either as `%uvk` line magic or as `%%cell` magic.
The difference is that the former puts out a new code cell with blank
metadata altered by the given options; the latter takes in script metadata
in the cell underneath it and *modifies it in place* according to the
given options, removing the top `%%uvk` line.

The simplest usage of this magic is all by itself as a line magic to
add a cell of blank script metadata, with a Python version constraint
requiring an interpreter version corresponding to the running interpreter's
or newer.

```ipython
%uvk
-------------------------------------------------------------------------
# /// script
# require-python = ">=3.13"
# dependencies = []
# ///
```

### Options

#### `-p`, `--python` *version-constraints*

Set the `require-python` field of the script metadata so that the Python
interpreter that will be chosen to run the kernel satisfies the given
constraints. Quote the specifier to use spaces.

Example:

```ipython
%uvk -p ">= 3.11, < 3.13, != 3.12.2"
-------------------------------------------------------------------------
# /// script
# require-python = "!=3.12.2,<3.13,>=3.11"
# dependencies = []
# ///
```

#### `-a`, `--add` *requirement* ...

Add the given package requirements to the `dependencies` field of the
metadata. The requirements are package names that one may suffix with
version constraints. If one of the specifications refers to a package that
is already required amongst the notebook's dependencies, the new
requirement clobbers the old one.

One can any number of requirements following the option; the listing
ends at the end of the line, of when another option leader is encountered.

Example:

```ipython
%%uvk -a requests "lark >1"
# /// script
# require-python = ">=3.11"
# dependencies = []
# ///
-------------------------------------------------------------------------
# /// script
# require-python = ">=3.11"
# dependencies = [
#     "requests",
#     "lark>1",
# ]
# ///
```

#### `-r`, `--remove` *package* ...

Remove the named packages from the `dependencies` list. While only
package names are required, one may also suffix constraints, but these
will be ignored (they still will cause errors if they are incorrectly
formatted). Named packages not already part of the dependencies are
silently ignored. Like the `--add` option, any number of packages can be
named. The list stops at the next option leader, or at the end of the
line.

Example:

```ipython
%%uvk -r lark
# /// script
# require-python = ">=3.11"
# dependencies = [
#     "requests",
#     "lark>1",
# ]
# ///
-------------------------------------------------------------------------
# /// script
# require-python = ">=3.11"
# dependencies = [
#     "requests",
# ]
# ///
```

#### `-A`, `--uv-args` *option* ... `--`

Specifies command-line arguments to add to the invocation of `uv`
that starts the IPython kernel for this notebook. Quoted arguments
are taken as is, not split as if unquoted. Any number of arguments can
be provided. The enumeration is presumed to stop at the `--` token, or at
the end of the line.

Example:

```ipython
%%uvk --uv-args --isolated --no-cache-dir -- --add numpy
# /// script
# require-python = ">=3.11"
# dependencies = []
# ///
-------------------------------------------------------------------------
# /// script
# require-python = ">=3.11"
# dependencies = [
#     "numpy",
# ]
#
# [tool.uvk]
# uv_args = [
#     "--isolated",
#     "--no-cache-dir",
# ]
# ///
```

#### `-h`, `--help`

Displays this document.

### Notes

Multiple options can be provided on the top line. One may fully
specify their metadata in a single operation:

```ipython
%uvk -p ">=3.11" -a requests numpy pandas<3 -A --index https://myindex.home/simple
----------------------------------------------------------------------------------
# /// script
# require-python = ">=3.11"
# dependencies = [
#     "requests",
#     "numpy",
#     "pandas<3",
# ]
#
# [tool.uvk]
# uv_args = [
#     "--index",
#     "https://myindex.home/simple",
# ]
# ///
```

This tool is not strictly necessary to edit the notebook's script
metadata so that <span style="font-family: sans-serif;">uvk</span>
takes it into account when starting the kernel. It is offered as a
mean of convenience. Script metadata typos might prevent the kernel
from starting, and Jupyterhub platform, no feedback is provided anywhere
visible when such difficulties occur.
